In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import time
import os

from src.datahandlers import ActivatingDataset
from src.utils import tuple_str_to_tuple
from src.neuron_explain import generate_explanation_prompt_dict

In [3]:
head_attributions_folder = "4_new_head_attributions"
suffix = ""
if head_attributions_folder == "4_head_attributions":
    suffix += "_baseline"

judge_model = "gpt-5.1"

In [4]:

filename = "2026-06-25_17-53-49_gpt2-small_train" 

with open(f'../experiment_data/{head_attributions_folder}/{filename}.json') as f:
    head_attributions = json.load(f)

trimmed_texts_filename = head_attributions['prior_filename']
with open(f'../experiment_data/3_trimmed_texts/{trimmed_texts_filename}.json') as f:
    trimmed_texts = json.load(f)

max_activating_filename = trimmed_texts['prior_filename']
with open(f'../experiment_data/2_max_activating_texts/{max_activating_filename}.json') as f:
    max_activating = json.load(f)

neuron_filename = max_activating['prior_filename']
with open(f'../experiment_data/1_next_token_neurons/{neuron_filename}.json') as f:
    neurons_data = json.load(f)

neurons = [tuple_str_to_tuple(neuron_str) for neuron_str in head_attributions['head_attributions'].keys()]

neuron_to_token = {tuple_str_to_tuple(neuron_str): token_data['token'] for neuron_str, token_data in neurons_data['neurons'].items()}

In [5]:
gpt_4_prompts_dict, nh_to_pos_neg_prompts = generate_explanation_prompt_dict(
    head_attribution_dict={tuple_str_to_tuple(k):v for k,v in head_attributions['head_attributions'].items()},
    neurons=neurons,
    neuron_to_token=neuron_to_token
)

In [ ]:
lengths = []

for v in nh_to_pos_neg_prompts.values():
    lengths.append(len(v[0])+len(v[1]))

mean_length = sum(lengths)/len(lengths)
print(mean_length)

16.021538461538462


In [7]:
parameters = head_attributions['parameters']
model_name = parameters['model_name']

output = {
    'parameters': parameters,
    'nh_to_pos_neg_prompts': {str(k): v for k,v in nh_to_pos_neg_prompts.items()},
    'prior_filename': filename,
}

# Save json to ../experiment_data/2_max_activating_texts
timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
new_filename = f"{timestamp}_{model_name}_train_prompts.json"

with open(f'../experiment_data/{head_attributions_folder}/{new_filename}', 'w') as f:
    json.dump(output, f)

In [8]:
jobs = [
            {"model":judge_model,
            "messages":[{"role": "user", "content": gpt_4_prompt}],
            "reasoning_effort": "none",
            "max_completion_tokens" : 200
        } for gpt_4_prompt in gpt_4_prompts_dict.values()]
 
filepath = f"../experiment_data/6_head_explanations/{filename + suffix}_{judge_model}.jsonl"
os.makedirs("../experiment_data/6_head_explanations", exist_ok=True)
if os.path.isfile(filepath):
    raise Exception("File already exists!")

with open(filepath, "w") as f:
    for job in jobs:
        json_string = json.dumps(job)
        f.write(json_string + "\n")

In [9]:
gpt_4_prompts_dict_str = {str(k):v for k,v in gpt_4_prompts_dict.items()}
with open(f"../experiment_data/6_head_explanations/{filename + suffix}_{judge_model}_prompts_dict.json", "w") as f:
    json.dump(gpt_4_prompts_dict_str, f)

In [10]:
prompts = list(gpt_4_prompts_dict.values())

string = ""
for prompt in prompts:
    string += prompt + "\n\n"

with open(f"../experiment_data/6_head_explanations/{filename + suffix}_{judge_model}_prompts.txt", "w") as f:
    f.write(string)

In [11]:
import tiktoken

prompts = list(gpt_4_prompts_dict.values())
encoder = tiktoken.get_encoding("o200k_base") # Encoding for gpt-5.1 
encoded_prompts = [encoder.encode(prompt) for prompt in prompts]

sum([len(x) for x in encoded_prompts])
encoder

<Encoding 'o200k_base'>